# InsightVLM — NB1: Baselines, Main Model, Full Evaluation

**This notebook alone produces every headline number the paper needs.**
Independent of NB2/NB3/NB4 — it holds no state from them and they hold none from it.


Output: `nb1_results.json`, `fig1_confusion.png`, `fig2_per_class_f1.png`,
`fig3_difficulty_provenance.png`, `results_table.tex`

In [ ]:
import os, sys, json, time, random, warnings, gc
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.models as M

SEED = 20260815
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.device_count(), "GPU(s)")

DATA_ROOT = Path("/kaggle/input/datasets/asifxzaman/xxxyyy/dataset/dataset")
def find_data():
    if (DATA_ROOT / "records.jsonl").exists(): return DATA_ROOT
    c = list(Path("/kaggle/input").glob("**/records.jsonl"))
    if not c: raise FileNotFoundError("records.jsonl not found under /kaggle/input")
    return c[0].parent
DATA = find_data(); print("dataset:", DATA)

meta = pd.read_parquet(DATA / "tables" / "metadata.parquet")
_ci = json.loads((DATA / "class_index.json").read_text())["label_to_id"]
CLASSES = sorted(_ci, key=_ci.get); NC = len(CLASSES); C2I = {c: i for i, c in enumerate(CLASSES)}
tr = meta[meta.split == "train"].reset_index(drop=True)
va = meta[meta.split == "val"].reset_index(drop=True)
te = meta[meta.split == "test"].reset_index(drop=True)
print(f"{len(meta):,} samples | {NC} classes | train {len(tr):,} val {len(va):,} test {len(te):,}")


NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def tf_train(size):

    return T.Compose([
        T.Resize((size, size)),
        T.RandomApply([T.ColorJitter(0.25, 0.25, 0.2, 0.03)], p=0.7),
        T.RandomApply([T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.95, 1.05))], p=0.5),
        T.RandomApply([T.GaussianBlur(3, (0.1, 1.2))], p=0.2),
        T.ToTensor(), NORM, T.RandomErasing(p=0.15, scale=(0.01, 0.04))])

def tf_eval(size):
    return T.Compose([T.Resize((size, size)), T.ToTensor(), NORM])

class ChartDS(Dataset):
    def __init__(self, df, size, train=False, crop_plot=False):
        self.df = df.reset_index(drop=True)
        self.tf = tf_train(size) if train else tf_eval(size)
        self.crop = crop_plot
        self.boxes = {}
        if crop_plot:
            for sp in ("train", "val", "test"):
                with open(DATA / "splits" / f"{sp}.jsonl") as fh:
                    for line in fh:
                        r = json.loads(line)
                        b = next((x["bbox"] for x in r["render"]["grounding_boxes"]
                                  if x["label"] == "plot_area"), None)
                        if b: self.boxes[r["sample_id"]] = b
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(DATA / r.image).convert("RGB")
        if self.crop:
            b = self.boxes.get(r.sample_id)
            if b:
                x0, y0, x1, y1 = [int(round(v)) for v in b]
                if x1 - x0 > 20 and y1 - y0 > 20: img = img.crop((x0, y0, x1, y1))
        return self.tf(img), C2I[r.label], i

def loader(df, size, train, bs=48, crop=False):
    # modest workers, no pinned memory, no persistent workers: the combination
    # of persistent_workers + pin_memory leaked ~5 GB per model
    return DataLoader(ChartDS(df, size, train, crop), batch_size=bs, shuffle=train,
                      num_workers=2, pin_memory=False, prefetch_factor=2,
                      persistent_workers=False)

def build(name, n=NC):
    if name == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "resnet101":
        m = M.resnet101(weights=M.ResNet101_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "densenet121":
        m = M.densenet121(weights=M.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, n)
    elif name == "resnet18":
        m = M.resnet18(weights=M.ResNet18_Weights.IMAGENET1K_V1); m.fc = nn.Linear(m.fc.in_features, n)
    else: raise ValueError(name)
    return m

def macro_f1(P, Y, n=NC):
    fs = []
    for i in range(n):
        tp = ((P==i)&(Y==i)).sum(); fp = ((P==i)&(Y!=i)).sum(); fn = ((P!=i)&(Y==i)).sum()
        pr = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0
        fs.append(2*pr*rc/(pr+rc) if pr+rc else 0)
    return float(np.mean(fs))

@torch.no_grad()
def predict(model, dl):
    model.eval(); P=[]; Y=[]; IX=[]
    for x, y, ix in dl:
        with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
            o = model(x.to(DEVICE, non_blocking=True))
        P.append(o.float().argmax(1).cpu()); Y.append(y); IX.append(ix)
    return torch.cat(P).numpy(), torch.cat(Y).numpy(), torch.cat(IX).numpy()

def train_model(name, size, epochs, bs=48, crop=False, patience=4, tag=""):
    dl_tr, dl_va = loader(tr, size, True, bs, crop), loader(va, size, False, bs, crop)
    model = build(name).to(DEVICE)
    if DEVICE.type == "cuda" and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, epochs=epochs,
                                                steps_per_epoch=len(dl_tr), pct_start=0.25)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type=="cuda")
    ckpt = f"{name}{tag}_best.pt"; best = -1.0; stale = 0; hist = []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot=n=corr=0
        for bi, (x, y, _) in enumerate(dl_tr):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.detach().item()*y.size(0); corr += (out.detach().argmax(1)==y).sum().item(); n += y.size(0)
            if bi % 60 == 0:
                print(f"\r  ep{ep} [{bi}/{len(dl_tr)}] loss {tot/max(n,1):.3f} acc {corr/max(n,1):.3f}", end="", flush=True)
        P, Y, _ = predict(model, dl_va); vf1 = macro_f1(P, Y)
        hist.append(dict(epoch=ep, loss=tot/n, train_acc=corr/n, val_acc=float((P==Y).mean()), val_f1=vf1))
        print(f"\r  ep{ep}  loss {tot/n:.3f}  train {corr/n:.3f}  val_acc {(P==Y).mean():.4f}  val_F1 {vf1:.4f}  ({time.time()-t0:.0f}s)")
        if vf1 > best:
            best, stale = vf1, 0
            torch.save((model.module if isinstance(model, nn.DataParallel) else model).state_dict(), ckpt)
        else:
            stale += 1
            if stale >= patience:
                print(f"  early stop: val macro-F1 flat for {patience} epochs"); break
    clean = build(name).to(DEVICE); clean.load_state_dict(torch.load(ckpt)); clean.eval()
    del model, dl_tr, dl_va, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()
    return clean, hist, best, ckpt

def save(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", path)
print("preamble ready")

## 1. Statistical oracle — the ceiling

In [ ]:
!pip -q install diptest pymannkendall ruptures

In [ ]:
import importlib.util
missing = [m for m in ("diptest","pymannkendall","ruptures","statsmodels")
           if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(f"taxonomy.py needs {missing}; it has SILENT fallbacks, so the "
                      f"oracle would report a wrong number instead of failing. "
                      f"Run the pip cell, then restart the kernel.")

TAX = None
for c in [Path("/kaggle/input/datasets/asifxzaman/xxxyyy/taxonomy.py")] + list(Path("/kaggle/input").glob("**/taxonomy.py")):
    if not c.exists(): continue
    sys.path.insert(0, str(c.parent)); import taxonomy as TAX; print("taxonomy:", c); break
if TAX is None: raise FileNotFoundError("taxonomy.py not attached")

_r = np.random.default_rng(0)
_x = np.r_[_r.normal(-3,.7,200), _r.normal(3,.7,200)]
assert TAX.verify_univariate(_x)[0] == "dist_bimodal", "diptest missing -- restart the kernel"
print("taxonomy self-check passed")

VERIFY = {"univariate": lambda d: TAX.verify_univariate(d["x"]),
          "bivariate":  lambda d: TAX.verify_bivariate(d["x"], d["y"]),
          "timeseries": lambda d: TAX.verify_timeseries(d["t"], d["y"]),
          "grouped":    lambda d: TAX.verify_grouped(d["value"], d["group"]),
          "composition":lambda d: TAX.verify_composition(d["category"])}
ok = sum(VERIFY[r.arity](json.loads(r.data_table_json))[0] == r.label for _, r in te.iterrows())
ORACLE = ok/len(te)
print(f"ORACLE accuracy: {ORACLE:.4f} ({ok}/{len(te)}) -- the ceiling")

## 2. Trivial floors

The 5% majority baseline **overstates** results on this task: chart type
correlates with the finding by construction (a pie chart can only be a
composition class). The chart-type prior uses no pixels and is the honest floor.

In [ ]:
floors = {"majority": float((te.label == tr.label.value_counts().idxmax()).mean())}
prior = tr.groupby("chart_type")["label"].agg(lambda s: s.value_counts().idxmax())
pred_ct = te.chart_type.map(prior).fillna(tr.label.value_counts().idxmax())
floors["chart_type_prior"] = float((pred_ct.values == te.label.values).mean())
for k, v in floors.items(): print(f"  {k:20s} {v:.4f}")
print(f"  {'oracle (ceiling)':20s} {ORACLE:.4f}")
print("\nA vision model is only interesting between the prior and the oracle.")

## 3. Main model — ResNet-50 @ 224

In [ ]:

MODEL, SIZE, EPOCHS = "resnet101", 224, 12
CKPT_HINTS = [f"{MODEL}_224_best.pt", f"{MODEL}_224_main_best.pt", f"{MODEL}_main_best.pt"]

found = None
for pat in CKPT_HINTS:
    hits = list(Path("/kaggle/input").glob(f"**/{pat}")) if Path("/kaggle/input").exists() else []
    if hits: found = hits[0]; break

if found:
    print(f"loading checkpoint: {found}")
    cnn = build(MODEL).to(DEVICE)
    cnn.load_state_dict(torch.load(found, map_location=DEVICE)); cnn.eval()
    hist, best_val = [], float("nan")
    print("skipped training -- evaluating the account-1 checkpoint")
else:
    print(f"no checkpoint found for {MODEL}; training from scratch (~45-60 min)")
    print("  (to skip this, upload results/account1/*.pt as a Kaggle dataset and attach it)")
    t0 = time.time()
    cnn, hist, best_val, _ = train_model(MODEL, SIZE, EPOCHS, tag="_main")
    print(f"\ntrained in {(time.time()-t0)/60:.1f} min | best val macro-F1 {best_val:.4f}")

P, Y, IX = predict(cnn, loader(te, SIZE, False))
ACC, F1 = float((P==Y).mean()), macro_f1(P, Y)
print(f"TEST accuracy {ACC:.4f}  macro-F1 {F1:.4f}")

## 4. Evaluation — per-class, per-difficulty, real-only probe

In [ ]:
res = pd.DataFrame({
    "true":[CLASSES[i] for i in Y], "pred":[CLASSES[i] for i in P],
    "difficulty":te.difficulty.values[IX], "provenance":te.provenance.values[IX],
    "family":te.family.values[IX], "chart_type":te.chart_type.values[IX]})
res["ok"] = res.true == res.pred

def boot_ci(mask, n=4000):
    a = np.asarray(mask, float); rng = np.random.default_rng(0)
    return tuple(np.percentile(a[rng.integers(0,len(a),(n,len(a)))].mean(1), [2.5,97.5]))

lo, hi = boot_ci(res.ok)
print(f"TEST accuracy {res.ok.mean():.4f}  [95% CI {lo:.4f}, {hi:.4f}]  macro-F1 {F1:.4f}\n")
for k in ("family","difficulty","provenance"):
    g = res.groupby(k).ok.agg(["mean","count"]).round(4); g.columns=["accuracy","n"]
    print(f"--- by {k} ---"); display(g)

probe = res[res.provenance=="real"]
plo, phi = boot_ci(probe.ok)
print(f"\nREAL-ONLY PROBE  n={len(probe)}  accuracy {probe.ok.mean():.4f} [95% CI {plo:.4f}, {phi:.4f}]")
print(f"synthetic-only   accuracy {res[res.provenance=='synthetic'].ok.mean():.4f}")
print("\nA large gap means the model learned the generator, not the statistics.")

## 5. Figures and LaTeX table

In [ ]:
cm = pd.crosstab(res.true, res.pred).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
cmn = cm.div(cm.sum(1).replace(0,1), axis=0)
fig, ax = plt.subplots(figsize=(11,9))
im = ax.imshow(cmn.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NC)); ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
ax.set_yticks(range(NC)); ax.set_yticklabels(CLASSES, fontsize=7)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{MODEL} @{SIZE}px")
for i in range(NC):
    for j in range(NC):
        v = cmn.values[i,j]
        if v > 0.02: ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=5.5,
                             color="white" if v>0.5 else "black")
fig.colorbar(im, shrink=0.8); plt.tight_layout()
plt.savefig("fig1_confusion.png", dpi=200, bbox_inches="tight"); plt.show()

print("most confused pairs:")
off = cmn.copy(); np.fill_diagonal(off.values, 0)
for (t,p), v in off.stack().sort_values(ascending=False).head(8).items():
    print(f"  {t:30s} -> {p:30s} {v:.3f}")

In [ ]:
f1_by = {}
for i,c in enumerate(CLASSES):
    tp=((P==i)&(Y==i)).sum(); fp=((P==i)&(Y!=i)).sum(); fn=((P!=i)&(Y==i)).sum()
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    f1_by[c]=2*pr*rc/(pr+rc) if pr+rc else 0
s = pd.Series(f1_by).sort_values()
fig, ax = plt.subplots(figsize=(7,6))
ax.barh(range(len(s)), s.values,
        color=["#c0392b" if v<.5 else "#e67e22" if v<.75 else "#27ae60" for v in s.values])
ax.set_yticks(range(len(s))); ax.set_yticklabels(s.index, fontsize=7)
ax.axvline(s.mean(), ls="--", c="k", lw=1, label=f"macro-F1 {s.mean():.3f}")
ax.set_xlabel("F1"); ax.legend(); plt.tight_layout()
plt.savefig("fig2_per_class_f1.png", dpi=200, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1,2, figsize=(11,3.6))
d = res.groupby("difficulty").ok.mean().reindex(["easy","medium","hard"])
axes[0].plot(d.index, d.values, "o-", lw=2); axes[0].set_ylim(0,1)
axes[0].set_ylabel("accuracy"); axes[0].set_title("Accuracy vs difficulty")
pv = res.groupby("provenance").ok.mean()
axes[1].bar(pv.index, pv.values, color=["#2980b9","#95a5a6"]); axes[1].set_ylim(0,1)
axes[1].set_title("Accuracy by provenance")
for i,v in enumerate(pv.values): axes[1].text(i, v+.02, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.savefig("fig3_difficulty_provenance.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
summary = pd.DataFrame([
    {"Model":"Majority class","Accuracy":floors["majority"],"Macro-F1":np.nan},
    {"Model":"Chart-type prior (no pixels)","Accuracy":floors["chart_type_prior"],"Macro-F1":np.nan},
    {"Model":f"{MODEL} @{SIZE}px","Accuracy":ACC,"Macro-F1":F1},
    {"Model":"Statistical oracle (ceiling)","Accuracy":ORACLE,"Macro-F1":np.nan},
]).round(4)
display(summary)
tex = summary.to_latex(index=False, na_rep="--", float_format="%.4f",
        caption="Baselines on InsightVLM-20K (test split, 4{,}000 samples).",
        label="tab:baselines")
Path("results_table.tex").write_text(tex); print(tex)

save({"oracle":ORACLE, "floors":floors,
      "model":{"name":MODEL,"size":SIZE,"epochs":EPOCHS,"accuracy":ACC,"macro_f1":F1,
               "ci95":[lo,hi],"best_val_f1":best_val,"history":hist},
      "by_difficulty":res.groupby("difficulty").ok.mean().round(4).to_dict(),
      "by_provenance":res.groupby("provenance").ok.mean().round(4).to_dict(),
      "by_family":res.groupby("family").ok.mean().round(4).to_dict(),
      "per_class_f1":{k:round(v,4) for k,v in f1_by.items()},
      "real_only_probe":{"n":int(len(probe)),"accuracy":float(probe.ok.mean()),"ci95":[plo,phi]}},
     "nb1_results.json")
res.to_csv("nb1_test_predictions.csv", index=False)
print("\nNB1 complete. Every headline number for the paper is in nb1_results.json.")